In [1]:
# FILE: 07_Hybrid_Pipeline.ipynb

import torch
from transformers import AutoProcessor, AutoModelForCausalLM, AutoConfig
from huggingface_hub import hf_hub_download
from openai import OpenAI
import cv2
import numpy as np
from PIL import Image
import os

# --- 1. CONFIGURATION ---
OPENROUTER_API_KEY = "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" # <--- PASTE YOUR KEY HERE
GIT_MODEL_NAME = "microsoft/git-base-vatex"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD GIT (The Eyes) ---
print("Loading Vision Model...")
# (Using the same patched loading logic as before to be safe)
checkpoint_path = hf_hub_download(repo_id=GIT_MODEL_NAME, filename="pytorch_model.bin")
state_dict = torch.load(checkpoint_path, map_location="cpu")
new_state_dict = {}
for k, v in state_dict.items():
    if "temperal" in k: new_state_dict[k.replace("temperal", "temporal")] = v
    else: new_state_dict[k] = v
keys_to_remove = ["git.embeddings.position_ids", "git.image_encoder.vision_model.embeddings.position_ids"]
for k in keys_to_remove: 
    if k in new_state_dict: del new_state_dict[k]

config = AutoConfig.from_pretrained(GIT_MODEL_NAME)
git_model = AutoModelForCausalLM.from_config(config)
git_model.load_state_dict(new_state_dict)
git_model = git_model.to(DEVICE)
git_processor = AutoProcessor.from_pretrained(GIT_MODEL_NAME)
print("✅ Vision Model Loaded.")

# --- 3. LOAD LLM CLIENT (The Brain) ---
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# --- 4. FUNCTIONS ---
def get_visual_fact(video_path):
    # Extract frames
    cap = cv2.VideoCapture(video_path)
    frames = []
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total > 0:
        indices = np.linspace(0, total-1, 6).astype(int)
        for i in range(total):
            ret, frame = cap.read()
            if i in indices and ret:
                frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    cap.release()
    
    # Generate Caption
    if not frames: return None
    inputs = git_processor(images=[frames], return_tensors="pt")
    pixel_values = inputs.pixel_values.to(DEVICE)
    ids = git_model.generate(pixel_values=pixel_values, max_length=50, num_beams=4)
    return git_processor.batch_decode(ids, skip_special_tokens=True)[0]

def get_styled_commentary(visual_fact, action_label):
    prompt = f"""
    You are a high-energy sports commentator.
    
    1. THE FACTS (What the camera sees): "{visual_fact}"
    2. THE CONTEXT (The game): "{action_label}"
    
    Task: Rewrite the factual description into a short, exciting spoken commentary line (max 15 words).
    Do NOT lie about what is happening (e.g. don't say "stadium" if it is "outside").
    """
    
    completion = client.chat.completions.create(
        model="meta-llama/llama-3.2-3b-instruct:free",
        messages=[{"role": "user", "content": prompt}]
    )
    return completion.choices[0].message.content.replace('"', '')

# --- 5. RUN THE HYBRID TEST ---
# Pick a video
test_path = "data/UCF50/Basketball/v_Basketball_g22_c03.avi" # The one you just tested

print(f"\n🎥 Analyzing Video...")
fact = get_visual_fact(test_path)
print(f"🔹 Visual Fact: {fact}")

print(f"🧠 Asking LLM to style it...")
commentary = get_styled_commentary(fact, "Basketball")
print(f"🎙️ Final Commentary: {commentary}")

C:\Users\rajam\miniconda3\envs\video_fusion\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading Vision Model...


C:\Users\rajam\AppData\Local\Temp\ipykernel_6172\3504139767.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(checkpoint_path, map_location="cpu")

✅ Vision Model Loaded.

🎥 Analyzing Video...
🔹 Visual Fact: two boys play basketball on a basketball court in a park.
🧠 Asking LLM to style it...
🎙️ Final Commentary: WOOHOO! Youngbloods duking it out on the outdoor hardwood in this rivalry showdown!
